# Seaborn Titanic 데이터셋 이진 분류 모델
이 노트북은 **TensorFlow/Keras**를 사용하여 타이타닉 생존 여부(survived)를 예측하는 딥러닝 모델을 구축하고 평가합니다.

In [ ]:
# 1. 라이브러리 임포트
import pandas as pd
import numpy as np
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

In [ ]:
# 2. 데이터 로드
df = sns.load_dataset('titanic')
print(df.head())

In [ ]:
# 3. 데이터 전처리
# 주요 특성 선택 및 결측치 처리
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

# 범주형 데이터 원-핫 인코딩
df = pd.get_dummies(df, columns=['sex', 'embarked'], drop_first=True, dtype=float)

# 데이터 분리
X = df.drop('survived', axis=1).values.astype(np.float32)
y = df['survived'].values.astype(np.float32)

# 학습/테스트 분할
np.random.seed(42)
indices = np.random.permutation(len(X))
split = int(len(X) * 0.8)
X_train, X_test = X[indices[:split]], X[indices[split:]]
y_train, y_test = y[indices[:split]], y[indices[split:]]

In [ ]:
# 4. Keras 모델 구조 정의 (로지스틱 회귀와 유사한 구조)
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid') # 이진 분류를 위한 시그모이드 출력
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
# 5. 모델 학습 (EarlyStopping 포함)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_split=0.2, epochs=100, batch_size=16, callbacks=[early_stopping], verbose=1)

In [ ]:
# 6. 학습 손실(Loss) 차트 시각화
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss History')
plt.legend()
plt.show()

In [ ]:
# 7. 오차 행렬(Confusion Matrix) 시각화
from sklearn.metrics import confusion_matrix
y_pred = (model.predict(X_test) > 0.5).astype(int)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=['Dead', 'Survived'], yticklabels=['Dead', 'Survived'])
plt.title('Confusion Matrix')
plt.show()